## Popcorn Hack: Fill in the Blanks

1. The **checkbox** on a toggle keeps track of whether it is on or off.
2. The **slider** is the visible part of the toggle that changes color when it is turned on.
3. The `.switch` class controls the toggle's position, size, padding, and **alignment**.
4. The **border-radius** property makes the slider have rounded edges.
5. The `&before` class dictates the **white circle** of the toggle.

## Homework: Safe Passage Heals Page Settings

Three toggles built with the `ocs__toggle` classes. Each one shows or hides a real section of the page, and a counter shows how many are on.

In [ ]:
%%html
<!-- UI_RUNNER: Toggles Homework | show_code: true -->

<div id="sph-settings">
  <!-- Each toggle = label (container) + checkbox (stores on/off) + track (visible switch) + label text -->
  <label class="ocs__toggle ocs__toggle--green">
    <input id="sph-toggle-hotline" class="ocs__toggle-input" type="checkbox" data-target="sph-hotline">
    <span class="ocs__toggle-track" aria-hidden="true"></span>
    <span class="ocs__toggle-label">Show support hotline</span>
  </label>

  <label class="ocs__toggle ocs__toggle--green">
    <input id="sph-toggle-resources" class="ocs__toggle-input" type="checkbox" data-target="sph-resources">
    <span class="ocs__toggle-track" aria-hidden="true"></span>
    <span class="ocs__toggle-label">Show resource categories</span>
  </label>

  <label class="ocs__toggle ocs__toggle--green">
    <input id="sph-toggle-tips" class="ocs__toggle-input" type="checkbox" data-target="sph-tips">
    <span class="ocs__toggle-track" aria-hidden="true"></span>
    <span class="ocs__toggle-label">Show warning signs</span>
  </label>

  <!-- aria-live="polite" makes screen readers announce the count when it changes -->
  <p><output id="sph-count" aria-live="polite">Toggles on: 0 of 3</output></p>

  <!-- The sections each toggle controls. "hidden" hides them until their toggle is on. -->
  <section id="sph-hotline" hidden>
    <h4>Need help now?</h4>
    <p>National Domestic Violence Hotline: 1-800-799-7233</p>
  </section>

  <section id="sph-resources" hidden>
    <h4>Resource categories</h4>
    <ul>
      <li>Counseling</li>
      <li>Housing</li>
      <li>Legal help</li>
    </ul>
  </section>

  <section id="sph-tips" hidden>
    <h4>Warning signs in a friend's relationship</h4>
    <ul>
      <li>Partner checks their phone or controls who they see</li>
      <li>They seem afraid of upsetting their partner</li>
      <li>They pull away from friends and activities</li>
    </ul>
  </section>
</div>

<script>
  // One list holds all three toggles, so one loop handles them instead of three copies of the same code.
  const sphToggles = document.querySelectorAll("#sph-settings .ocs__toggle-input");
  const sphCount = document.getElementById("sph-count");

  function updateSettings() {
    let onCount = 0;

    sphToggles.forEach(function (toggle) {
      // data-target stores the id of the section this toggle controls,
      // so the HTML decides the pairing and the JavaScript stays the same.
      const section = document.getElementById(toggle.dataset.target);

      // I thought .checked was interesting: it's a true/false the browser keeps
      // updated, so JavaScript can just read it instead of tracking clicks itself.
      if (toggle.checked) {
        section.hidden = false;
        onCount = onCount + 1;
      } else {
        section.hidden = true;
      }
    });

    sphCount.textContent = "Toggles on: " + onCount + " of " + sphToggles.length;
  }

  // "change" fires every time a toggle flips, so the count and sections always match.
  sphToggles.forEach(function (toggle) {
    toggle.addEventListener("change", updateSettings);
  });

  updateSettings();
</script>

## How it works

```text
User flips toggle
      │
      ▼
checkbox.checked becomes true/false   (state lives in the input)
      │
      ▼
"change" event → updateSettings()
      │
      ├─ FOR EACH toggle:
      │     IF checked → show its section, add 1 to count
      │     ELSE       → hide its section
      │
      ▼
Counter shows "Toggles on: N of 3"
```

- **SASS** (`ocs__toggle` classes) handles how the switch looks
- **HTML** stores the state (checkbox) and the pairing (`data-target`)
- **JavaScript** reads the state and updates the page
- **College Board link:** iteration (`forEach`) + selection (`if/else`) in one procedure

## The SASS behind my toggles

The lesson says to use the OCS toggle classes on the page, so the HTML above has no custom CSS. This is the SASS that makes `ocs__toggle`, `ocs__toggle-input`, and `ocs__toggle-track` work, based on OCS `elements/toggles/common.scss`. It uses **boolean variables** (`$toggle-rounded`, `$toggle-animate`, `$toggle-show-focus`) with **`@if` / `@else`** to switch features on or off, a **mixin** for color variants like `--green`, and the `:checked` state to slide the thumb when a toggle is on.

```scss
// ---- Settings: boolean variables turn features on or off ----
$toggle-rounded: true;      // pill-shaped track vs. square track
$toggle-animate: true;      // slide the thumb, or jump instantly
$toggle-show-focus: true;   // keyboard focus ring (accessibility)

// ---- Size values (same numbers OCS uses) ----
$toggle-width: 54px;
$toggle-height: 31px;
$toggle-thumb-size: 23px;
$toggle-inset: 4px;
$toggle-travel: $toggle-width - $toggle-thumb-size - (2 * $toggle-inset); // how far the thumb slides

// ---- Mixin: one reusable rule for each color variant ----
@mixin toggle-accent($color) {
  --ocs-toggle-accent: #{$color};
}

// label.ocs__toggle = the clickable container
.ocs__toggle {
  @include toggle-accent(var(--pref-accent-color));
  display: inline-flex;
  align-items: center;
  gap: 10px;
  cursor: pointer;

  // Variant used in my homework: green toggles
  &--green {
    @include toggle-accent(var(--pref-success-color, var(--pref-accent-color)));
  }
}

// The real checkbox: hidden, but it stores on/off
.ocs__toggle-input {
  position: absolute;
  width: 1px;
  height: 1px;
  opacity: 0;

  // ON state: color the track and slide the thumb
  &:checked + .ocs__toggle-track {
    background: var(--ocs-toggle-accent);

    &::before {
      transform: translateX($toggle-travel);
    }
  }

  // Keyboard focus ring only if the setting is on
  @if $toggle-show-focus {
    &:focus-visible + .ocs__toggle-track {
      outline: 2px solid var(--ocs-toggle-accent);
      outline-offset: 2px;
    }
  }

  // Disabled: faded and not clickable
  &:disabled ~ .ocs__toggle-track,
  &:disabled ~ .ocs__toggle-label {
    cursor: not-allowed;
    opacity: 0.55;
  }
}

// The visible switch
.ocs__toggle-track {
  position: relative;
  width: $toggle-width;
  height: $toggle-height;
  background: var(--panel-mid);

  // @if / @else: choose the shape from the boolean setting
  @if $toggle-rounded {
    border-radius: $toggle-height;
  } @else {
    border-radius: 4px;
  }

  // The thumb (white circle)
  &::before {
    content: "";
    position: absolute;
    top: $toggle-inset;
    left: $toggle-inset;
    width: $toggle-thumb-size;
    height: $toggle-thumb-size;
    background: var(--pref-text-color);

    @if $toggle-rounded {
      border-radius: 50%;    // round thumb
    } @else {
      border-radius: 3px;    // square thumb
    }

    @if $toggle-animate {
      transition: transform 0.2s ease;
    } @else {
      transition: none;
    }
  }
}
```